# Выбор метрики для Bank Marketing

## Структура
1. Бизнес-контекст
2. Дисбаланс классов
3. 20 вопросов для выбора метрики
4. Бизнес-вопрос - метрика, как связаны
5. Роль baseline-модели
6. Выбор метрики
7. Почему не другие метрики
8. Ограничения выбранной метрики
9. Что скажем бизнесу
10. Итоговый вывод

## 1. Бизнес-контекст

### Что делает банк
Банк проводит исходящие телефонные кампании и предлагает клиентам оформить срочный вклад (депозит). Клиент может ответить `yes` (оформит) или `no` (не оформит)

### Что делает модель
Модель ранжирует клиентов по вероятности подписки. Менеджеры обзванивают клиентов сверху списка. Модель не звонит автоматически - решение принимает менеджер, но приоритет задаёт модель

### Ограничения бизнеса
- Бюджет кампании ограничен. Банк может обзвонить только K клиентов (например, 10% базы)
- Каждый звонок стоит денег (c_call - время менеджера и телефония)
- Каждый оформленный депозит приносит прибыль (p_deposit)
- Обычно p_deposit >> c_call (в 10-20 раз), потому что депозит - крупная сумма, а звонок - минуты работы менеджера

### Что важно для бизнеса
- Найти как можно больше клиентов, готовых оформить депозит
- Не тратить бюджет на пустые звонки
- Понимать, сколько денег приносит модель

## 2. Дисбаланс классов

В данных 88.73% ответов `no` и 11.27% ответов `yes`

Это сильный дисбаланс. Класс `yes` (клиенты, готовые оформить депозит) - редкий, но именно он важен для бизнеса

### Что это значит для выбора метрики
- Модель, которая всегда говорит `no`, даёт Accuracy около 88.7%. Это высокий показатель, но модель бесполезна: она не находит ни одного клиента
- Метрика должна делать дисбаланс заметным. Accuracy и ROC-AUC этого не делают
- Baseline для случайного предсказания на положительном классе = prevalence = 0.1127. Любая метрика, которая это не учитывает, будет обманывать

## 3. 20 вопросов для выбора метрики

### 1. Какое бизнес-решение принимает модель?
модель формирует список для обзвона. Это приоритизация: кому звонить первым. Не автоматическое решение

### 2. Каков доступный бюджет кампании?
ьюджет ограничен: банк может обзвонить только K клиентов (например, 10% базы)

### 3. Сколько стоит один звонок?
c_call - стоимость звонка. примем c_call = 1 условную единицу

### 4. Сколько приносит один оформленный депозит?
p_deposit - прибыль от одного депозита. примем p_deposit = 10 условных единиц, т.е. p_deposit >> c_call

### 5. Что дороже для банка: пропустить клиента или сделать лишний звонок?
пропустить клиента (FN) дороже, чем сделать лишний звонок (FP):
- FN = упущенная прибыль p_deposit
- FP = потраченный c_call
- p_deposit >> c_call, значит FN дороже FP

но precision тоже важен: нельзя тратить весь бюджет на пустые звонки

### 6. Какая допустимая доля ложных срабатываний?
FP ограничен бюджетом, если бюджет - K звонков, то FP = K - TP, т.е. модель должна максимизировать TP при фиксированном K

### 7. Какой минимально необходимый уровень охвата реальных клиентов?
ьанк хочет захватить хотя бы 60-70% реальных подписчиков в топ-K. Это требование к Recall@K

### 8. Нужны вероятности или ранжирование?
для ранжирования достаточно score, но калибровка вероятностей полезна, потому что позволяет считать expected profit для каждого клиента

### 9. Какой размер топ-K?
K = 10% базы (около 4000 клиентов из 41000) - это ограничение бюджета

### 10. Какая доля реальных подписчиков должна быть в топ-K?
целевой Recall@K ≥ 60%

### 11. Как измерять эффект на продакшене?
сравнивать с контрольной группой (random selection) и предыдущей кампанией. Основная бизнес-метрика - Lift@K (во сколько раз модель лучше случайного отбора)

### 12. Какая метрика для финального решения?
Lift@K и Expected Profit@K. PR-AUC - для сравнения моделей (интегральная характеристика), но финальное решение - по бизнес-метрикам

### 13. Насколько дисбалансированы классы?
88.7% / 11.3%. Метрика не должна делать дисбаланс незаметным (как Accuracy)

### 14. Нужна ли калибровка вероятностей?
для ранжирования - нет
для expected profit - да. Логистическая регрессия калибрована сразу, а LightGBM - нужно проверять

### 15. Учитывает ли метрика, что решение принимается только для топ-K?
Accuracy и ROC-AUC - нет
Precision@K, Recall@K, Lift@K, Expected Profit@K - да

### 16. Подходит ли метрика к временной структуре данных?
данные собраны за несколько лет. Идеально - time-based split? но можно StratifiedKFold

### 17. Не использует ли метрика информацию, недоступную в момент прогноза?
`duration` удалён (leak). `pdays` заменён на `was_contacted` (доступен в момент прогноза). PR-AUC использует только предсказания, всё норм

### 18. Можно ли интерпретировать метрику в бизнес-терминах?
PR-AUC - плохо (площадь под кривой). Lift@K и Expected Profit@K - хорошо, т.к. во сколько раз лучше случайного и сколько заработали

### 19. Какая метрика позволит сравнивать модели на одном наборе?
Precision@K, Recall@K, Lift@K - если K фиксирован
PR-AUC - тоже, но она интегральная

### 20. Какой порог или размер K зафиксирован?
K = 10% базы (фиксирован бюджетом). Порог классификации не фиксирован, а выбирается по Expected Profit

## 4. Бизнес-вопрос - метрика, как связаны

| бизнес-вопрос | что важно | метрика |
|---|---|---|
| кому звонить первым? | ранжирование | PR-AUC |
| сколько оформили из K обзвоненных? | точность на топ-K | Precision@K |
| сколько реальных клиентов поймали? | полнота на топ-K | Recall@K |
| сасколько лучше случайного? | эффект | Lift@K |
| сколько заработали? | деньги | Expected Profit@K |

каждая метрика отвечает на свой бизнес-вопрос. не выбираем одну метрику, а используем набор, где каждая юудет закрывать свой уровень решения

## 5. Роль baseline-модели

baseline-модель - Logistic Regression с `class_weight='balanced'`

### Что даст
- калиброванные вероятности (не только score). Значит, Expected Profit@K считается корректно
- интерпретируемость: видно, какие признаки важны (например, `was_contacted`), и это согласуется с бизнес-логикой
- отправную точку для сравнения: если LightGBM не лучше LR по Lift@K, то усложнять модель нет смысла

### Ограничения
LR - линейная модель, она не справляется со сложными взаимодействиями, поэтому метрика должна быть устойчивой к среднему качеству модели. PR-AUC это позволяет - он не требует идеального разделения классов, а измеряет качество на всём диапазоне порогов

### Что это значит для выбора метрики
- PR-AUC подходит, потому что не требует идеального качества модели
- Expected Profit@K подходит, потому что LR даёт калиброванные вероятности
- Accuracy и ROC-AUC не подходят, потому что обманывают при дисбалансе и не привязаны к бизнес-решению

## 6. Выбор метрики

| метрика | что измеряет | бизнес-смысл | роль в кейсе |
|---|---|---|---|
| **PR-AUC** | качество на положительном классе | - | сравнение моделей |
| **Precision@K** | точность на топ-K | сколько оформили из K обзвоненных | оценка бюджета |
| **Recall@K** | полнота на топ-K | сколько реальных клиентов поймали | бизнес-цель |
| **Lift@K** | во сколько раз лучше случайного | эффект модели | шлавная бизнес-метрика |
| **Expected Profit@K** | деньги | сколько заработали | финальная метрика |

### Главный вывод
- PR-AUC - для сравнения моделей (честная на дисбалансе, baseline = prevalence)
- Lift@K и Recall@K - для оценки бизнес-эффекта
- Expected Profit@K - финальная метрика для выбора порога и модели

## 7. Почему не другие метрики

### Accuracy
Accuracy не различает полезную модель и бесполезную константу

### ROC-AUC
при 36000 TN даже 100 FP дают FPR ≈ 0.003. Разница между хорошей и слабой моделью сглаживается

### F1
симметрична: одинаково штрафует FP и FN. В бизнесе FN дороже FP (p_deposit >> c_call), поэтому F1 не отражает экономику задачи

## 8. Ограничения выбранной метрики

- **PR-AUC зависит от prevalence.** нельзя сравнивать PR-AUC между разными датасетами, только внутри одного
- **Lift@K зависит от K.** при разных K - разные значения. K нужно фиксировать до сравнения моделей
- **Expected Profit зависит от C_call и P_deposit.** эти параметры - бизнес-допущения
- **PR-AUC не интерпретируется бизнесом напрямую.** он нужен для сравнения моделей, а не для отчёта бизнесу

## 9. Что скажем бизнесу

бизнесу не нужен PR-AUC. бизнесу нужно:

- "модель в 3 раза лучше случайного отбора" (Lift@10%)
- "из 4000 обзвоненных оформили 400, это +250 к случайному" (Precision@K)
- "ожидаемая прибыль - 2000 условных единиц" (Expected Profit@K)

PR-AUC - для дата-сайентистов (сравнить модели). Lift@K и Expected Profit@K - для бизнеса (принять решение)

это ключевое различие: техническая метрика и бизнес-метрика - разные вещи. используем обе, но для разных целей

## 10. Итоговый вывод

### Метрика выбрана по бизнес-задаче
- банк формирует список для обзвона с ограниченным бюджетом (K = 10%)
- FN дороже FP (p_deposit >> c_call), но precision тоже важен (бюджет ограничен)
- модель ранжирует клиентов, а не классифицирует всех
- baseline-модель - LR, она даёт калиброванные вероятности, поэтому Expected Profit@K считается корректно

### Какие метрики используем
| метрика | роль |
|---|---|
| **PR-AUC** | сравнение моделей (честная на дисбалансе, baseline = prevalence) |
| **Precision@K** | оценка бюджета (сколько оформили из K обзвоненных) |
| **Recall@K** | бизнес-цель (сколько реальных клиентов поймали в топ-K) |
| **Lift@K** | главная бизнес-метрика (во сколько раз лучше случайного) |
| **Expected Profit@K** | финальная метрика для выбора порога |

### Что не используем
- **Accuracy** - обманывает при дисбалансе
- **ROC-AUC** - сглаживает разницу
- **F1** - симметрична, не учитывает бизнес-стоимость ошибок
- **MCC, Balanced Accuracy** - учитывают TN, который нам не интересен